# Preprocesamiento de Vitales — Series Temporales

Extrae signos vitales de `chartevents` para los stays de la cohorte Sepsis-3.

**Salida**: `data/processed/vitals/vitals_hourly.parquet`  
Formato: una fila por `(stay_id, hours_from_intime)` con columnas por cada vital.

**Diseño de ventana**: almacenamos toda la estancia alineada a `intime`.  
La ventana de predicción [onset−48h, onset−6h/12h] se aplica en el entrenamiento del TFT.

| Variable | itemid | Unidad |
|---|---|---|
| heart_rate | 220045 | bpm |
| resp_rate | 220210 | resp/min |
| spo2 | 220277 | % |
| map_art | 220052 | mmHg |
| map_ni | 220181 | mmHg |
| temp_c | 223762 | °C (directo) |
| temp_f | 223761 | °F → °C (mayoritario; se convierte y consolida en `temp_c`) |
| fio2 | 223835 | % |
| gcs_eye | 220739 | — |
| gcs_verbal | 223900 | — |
| gcs_motor | 223901 | — |
| sbp_ni | 220179 | mmHg |
| dbp_ni | 220180 | mmHg |

> La temperatura se registra en MIMIC-IV predominantemente en Fahrenheit (223761). Usar solo el item Celsius (223762) dejaba ~94% de valores ausentes; se consolidan ambos en un único canal `temp_c` en grados Celsius.

In [1]:
import polars as pl
import numpy as np
from pathlib import Path
import time

MIMIC   = Path.home() / "mimic-iv-3.0"
ICU     = MIMIC / "icu"
OUT_DIR = Path("../data/processed/vitals")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COHORT_PATH = Path("../data/processed/cohort.parquet")

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

VITAL_ITEMS = {
    "heart_rate":  220045,
    "resp_rate":   220210,
    "spo2":        220277,
    "map_art":     220052,
    "map_ni":      220181,
    "temp_c":      223762,   # Temperatura en Celsius (registro directo, minoritario)
    "temp_f":      223761,   # Temperatura en Fahrenheit (mayoría de registros en MIMIC-IV)
    "fio2":        223835,
    "gcs_eye":     220739,
    "gcs_verbal":  223900,
    "gcs_motor":   223901,
    "sbp_ni":      220179,
    "dbp_ni":      220180,
}

ALL_ITEM_IDS = list(VITAL_ITEMS.values())

# Lookup DataFrame: itemid (Int64) → vital name
# FIX: usar join en vez de pl.col.replace(dict) que falla con Int64→String en Polars 1.x
item_lookup = pl.DataFrame({
    "itemid": list(VITAL_ITEMS.values()),
    "vital":  list(VITAL_ITEMS.keys()),
}).with_columns(pl.col("itemid").cast(pl.Int64))

print(f"Vitales a extraer: {len(VITAL_ITEMS)}")
print(item_lookup)

Vitales a extraer: 13
shape: (13, 2)
┌────────┬────────────┐
│ itemid ┆ vital      │
│ ---    ┆ ---        │
│ i64    ┆ str        │
╞════════╪════════════╡
│ 220045 ┆ heart_rate │
│ 220210 ┆ resp_rate  │
│ 220277 ┆ spo2       │
│ 220052 ┆ map_art    │
│ 220181 ┆ map_ni     │
│ …      ┆ …          │
│ 220739 ┆ gcs_eye    │
│ 223900 ┆ gcs_verbal │
│ 223901 ┆ gcs_motor  │
│ 220179 ┆ sbp_ni     │
│ 220180 ┆ dbp_ni     │
└────────┴────────────┘


## PASO 1 — Cargar cohorte

In [2]:
cohort = pl.read_parquet(COHORT_PATH).select([
    "subject_id", "hadm_id", "stay_id",
    "intime", "outtime", "los_hours",
    "sepsis", "onset_time",
])

stay_ids = cohort["stay_id"].to_list()

print(f"Stays en cohorte:  {len(cohort):,}")
print(f"Sepsis:            {cohort['sepsis'].sum():,}")
print(f"Schema: {cohort.schema}")
cohort.head(3)

Stays en cohorte:  74,829
Sepsis:            17,189
Schema: Schema([('subject_id', Int64), ('hadm_id', Int64), ('stay_id', Int64), ('intime', Datetime(time_unit='us', time_zone=None)), ('outtime', Datetime(time_unit='us', time_zone=None)), ('los_hours', Int64), ('sepsis', Int8), ('onset_time', Datetime(time_unit='us', time_zone=None))])


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 2 — Leer chartevents filtrado a la cohorte

In [3]:
log("Leyendo chartevents (puede tardar ~3-4 min) ...")

charts = (
    pl.read_csv(
        ICU / "chartevents.csv.gz",
        columns=["stay_id", "itemid", "charttime", "valuenum"],
    )
    .with_columns([
        pl.col("stay_id").cast(pl.Int64),
        pl.col("itemid").cast(pl.Int64),
        pl.col("charttime").str.to_datetime(strict=False),
    ])
    .filter(
        pl.col("itemid").is_in(ALL_ITEM_IDS) &
        pl.col("stay_id").is_in(stay_ids) &
        pl.col("valuenum").is_not_null()
    )
)

log(f"Registros cargados: {len(charts):,}")
log(f"Stays con datos:    {charts['stay_id'].n_unique():,}")
charts.head(5)

[19:35:26] Leyendo chartevents (puede tardar ~3-4 min) ...


[19:38:51] Registros cargados: 53,039,974


[19:38:51] Stays con datos:    74,829


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


In [4]:
# Verificar cobertura por vital
# FIX: join con item_lookup en vez de replace(dict) — evita InvalidOperationError Int64→String
coverage = (
    charts
    .join(item_lookup, on="itemid", how="left")
    .group_by("vital")
    .agg([
        pl.col("stay_id").n_unique().alias("stays_con_dato"),
        pl.col("valuenum").count().alias("n_registros"),
        pl.col("valuenum").mean().round(2).alias("media"),
        pl.col("valuenum").std().round(2).alias("std"),
    ])
    .with_columns(
        (pl.col("stays_con_dato") / len(cohort) * 100).round(1).alias("cobertura_pct")
    )
    .sort("cobertura_pct", descending=True)
)

print(f"Cobertura por vital (sobre {len(cohort):,} stays):")
print(coverage)

Cobertura por vital (sobre 74,829 stays):
shape: (13, 6)
┌────────────┬────────────────┬─────────────┬────────┬─────────┬───────────────┐
│ vital      ┆ stays_con_dato ┆ n_registros ┆ media  ┆ std     ┆ cobertura_pct │
│ ---        ┆ ---            ┆ ---         ┆ ---    ┆ ---     ┆ ---           │
│ str        ┆ u32            ┆ u32         ┆ f64    ┆ f64     ┆ f64           │
╞════════════╪════════════════╪═════════════╪════════╪═════════╪═══════════════╡
│ gcs_eye    ┆ 74805          ┆ 2114512     ┆ 3.26   ┆ 1.06    ┆ 100.0         │
│ gcs_motor  ┆ 74800          ┆ 2105099     ┆ 5.23   ┆ 1.5     ┆ 100.0         │
│ gcs_verbal ┆ 74803          ┆ 2110387     ┆ 3.11   ┆ 1.87    ┆ 100.0         │
│ heart_rate ┆ 74828          ┆ 8391447     ┆ 88.2   ┆ 3882.32 ┆ 100.0         │
│ spo2       ┆ 74813          ┆ 8221644     ┆ 104.17 ┆ 7822.03 ┆ 100.0         │
│ …          ┆ …              ┆ …           ┆ …      ┆ …       ┆ …             │
│ temp_f     ┆ 74113          ┆ 1969178     ┆ 98.71 

## PASO 3 — Alinear a intime y calcular hora relativa

`hours_from_intime = floor((charttime − intime) / 1h)`

In [5]:
log("Alineando a intime ...")

charts_aligned = (
    charts
    .join(
        cohort.select(["stay_id", "intime", "outtime", "los_hours"]),
        on="stay_id",
        how="left",
    )
    .with_columns(
        ((pl.col("charttime") - pl.col("intime")).dt.total_minutes() / 60)
        .floor().cast(pl.Int32).alias("hours_from_intime")
    )
    # Descartar medidas fuera de la estancia
    .filter(
        (pl.col("hours_from_intime") >= 0) &
        (pl.col("hours_from_intime") <= pl.col("los_hours").cast(pl.Int32))
    )
    # FIX: join con item_lookup en vez de replace(dict)
    .join(item_lookup, on="itemid", how="left")
    .select(["stay_id", "hours_from_intime", "vital", "valuenum"])
)

log(f"Registros alineados: {len(charts_aligned):,}")
charts_aligned.head(5)

[19:38:53] Alineando a intime ...


[19:38:55] Registros alineados: 52,884,871


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 4 — Remuestreo a 1h (media por bucket)

In [6]:
log("Remuestreando a 1h (media por bucket) ...")

charts_1h_long = (
    charts_aligned
    .group_by(["stay_id", "hours_from_intime", "vital"])
    .agg(pl.col("valuenum").mean().alias("value"))
)

charts_1h = (
    charts_1h_long
    .pivot(index=["stay_id", "hours_from_intime"], on="vital", values="value")
    .sort(["stay_id", "hours_from_intime"])
)

# Garantizar que existen todas las columnas de vitales
for col_name in VITAL_ITEMS.keys():
    if col_name not in charts_1h.columns:
        charts_1h = charts_1h.with_columns(pl.lit(None, dtype=pl.Float64).alias(col_name))

log(f"Filas en formato ancho (stay × hora): {len(charts_1h):,}")
log(f"Stays cubiertos: {charts_1h['stay_id'].n_unique():,}")
charts_1h.head(8)

[19:38:55] Remuestreando a 1h (media por bucket) ...


[19:39:02] Filas en formato ancho (stay × hora): 7,571,427
[19:39:02] Stays cubiertos: 74,829


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 5 — Forward-fill hasta 4h

Primero expandimos la grilla horaria completa (sin huecos) para cada stay, luego aplicamos ffill.  
Con más de 4h sin medida, el valor queda como `null`.

In [7]:
log("Expandiendo grilla horaria completa por stay ...")

stay_ranges = (
    cohort.select(["stay_id", "los_hours"])
    .with_columns(pl.col("los_hours").cast(pl.Int32))
)

hour_grid = (
    stay_ranges
    .with_columns(
        pl.int_ranges(0, pl.col("los_hours") + 1).alias("hours_from_intime")
    )
    .explode("hours_from_intime")
    .select(["stay_id", "hours_from_intime"])
)

log(f"Grilla completa: {len(hour_grid):,} filas")

vitals_full = (
    hour_grid
    .join(charts_1h, on=["stay_id", "hours_from_intime"], how="left")
    .sort(["stay_id", "hours_from_intime"])
)

log(f"Grilla con datos: {len(vitals_full):,} filas")
vitals_full.head(8)

[19:39:02] Expandiendo grilla horaria completa por stay ...
[19:39:02] Grilla completa: 7,940,236 filas


[19:39:03] Grilla con datos: 7,940,236 filas


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


In [8]:
log("Aplicando forward-fill (máx 4h) por stay ...")

vital_cols = list(VITAL_ITEMS.keys())

vitals_ffill = vitals_full.with_columns([
    pl.col(c).forward_fill(limit=4).over("stay_id")
    for c in vital_cols
])

null_pct = {
    c: round(100 * vitals_ffill[c].is_null().sum() / len(vitals_ffill), 1)
    for c in vital_cols
}
print("% nulos tras forward-fill (4h):")
for k, v in sorted(null_pct.items(), key=lambda x: x[1]):
    bar = '█' * int(v / 2)
    print(f"  {k:<14} {v:>5}%  {bar}")

[19:39:03] Aplicando forward-fill (máx 4h) por stay ...


% nulos tras forward-fill (4h):
  heart_rate       1.3%  
  spo2             1.7%  
  resp_rate        2.0%  █
  temp_f          13.0%  ██████
  gcs_eye         13.2%  ██████
  gcs_verbal      13.3%  ██████
  gcs_motor       13.4%  ██████
  sbp_ni          30.9%  ███████████████
  dbp_ni          30.9%  ███████████████
  map_ni          31.0%  ███████████████
  fio2            53.1%  ██████████████████████████
  map_art         64.9%  ████████████████████████████████
  temp_c          94.3%  ███████████████████████████████████████████████


## PASO 6 — MAP consolidado y GCS total

In [9]:
vitals_final = vitals_ffill.with_columns([
    pl.when(pl.col("map_art").is_not_null())
      .then(pl.col("map_art"))
      .otherwise(pl.col("map_ni"))
      .alias("map"),

    pl.when(
        pl.col("gcs_eye").is_not_null() &
        pl.col("gcs_verbal").is_not_null() &
        pl.col("gcs_motor").is_not_null()
    )
    .then(pl.col("gcs_eye") + pl.col("gcs_verbal") + pl.col("gcs_motor"))
    .otherwise(None)
    .alias("gcs_total"),

    # Consolidar temperatura: usar el registro en Celsius si existe; en su defecto,
    # convertir el registro en Fahrenheit (mayoritario en MIMIC-IV) a Celsius.
    # Antes se usaba solo el item Celsius (223762), con ~94% de valores ausentes.
    pl.when(pl.col("temp_c").is_not_null())
      .then(pl.col("temp_c"))
      .when(pl.col("temp_f").is_not_null())
      .then((pl.col("temp_f") - 32.0) * 5.0 / 9.0)
      .otherwise(None)
      .alias("temp_c"),
])

FINAL_COLS = [
    "stay_id", "hours_from_intime",
    "heart_rate", "resp_rate", "spo2",
    "map", "map_art", "map_ni",
    "sbp_ni", "dbp_ni",
    "temp_c", "fio2",
    "gcs_total", "gcs_eye", "gcs_verbal", "gcs_motor",
]

vitals_final = vitals_final.select(FINAL_COLS)

print(f"Shape final: {vitals_final.shape}")
vitals_final.head(5)

Shape final: (7940236, 16)


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 7 — Sanity checks y clipping de outliers

In [10]:
CLINICAL_RANGES = {
    "heart_rate": (10, 300),
    "resp_rate":  (4, 60),
    "spo2":       (50, 100),
    "map":        (20, 200),
    "temp_c":     (30, 42),
    "fio2":       (21, 100),
    "gcs_total":  (3, 15),
}

print("Outliers por variable (fuera de rango clínico):")
for col_name, (lo, hi) in CLINICAL_RANGES.items():
    if col_name not in vitals_final.columns:
        continue
    n_out = vitals_final.filter(
        pl.col(col_name).is_not_null() &
        ((pl.col(col_name) < lo) | (pl.col(col_name) > hi))
    ).height
    total = vitals_final[col_name].is_not_null().sum()
    pct = 100 * n_out / total if total > 0 else 0
    print(f"  {col_name:<14} {n_out:>6} outliers  ({pct:.2f}%)  [{lo}, {hi}]")

# Clipping
log("Aplicando clipping ...")
clip_exprs = [
    pl.when((pl.col(c) < lo) | (pl.col(c) > hi))
      .then(None)
      .otherwise(pl.col(c))
      .alias(c)
    for c, (lo, hi) in CLINICAL_RANGES.items()
    if c in vitals_final.columns
]
vitals_final = vitals_final.with_columns(clip_exprs)
log("Clipping aplicado.")

Outliers por variable (fuera de rango clínico):
  heart_rate       3098 outliers  (0.04%)  [10, 300]
  resp_rate       26298 outliers  (0.34%)  [4, 60]
  spo2             3523 outliers  (0.05%)  [50, 100]
  map             17238 outliers  (0.23%)  [20, 200]
  temp_c           5534 outliers  (0.08%)  [30, 42]
  fio2            14260 outliers  (0.38%)  [21, 100]
  gcs_total           0 outliers  (0.00%)  [3, 15]
[19:39:03] Aplicando clipping ...
[19:39:03] Clipping aplicado.


## PASO 8 — Unir etiquetas y guardar

In [11]:
log("Uniendo con etiquetas de cohorte ...")

cohort_labels = cohort.select(["stay_id", "intime", "sepsis", "onset_time"]).with_columns(
    pl.when(pl.col("onset_time").is_not_null())
    .then(
        ((pl.col("onset_time") - pl.col("intime")).dt.total_minutes() / 60)
        .floor().cast(pl.Int32)
    )
    .otherwise(None)
    .alias("onset_hour")
)

vitals_labeled = vitals_final.join(
    cohort_labels.select(["stay_id", "sepsis", "onset_hour"]),
    on="stay_id",
    how="left",
)

output_path = OUT_DIR / "vitals_hourly.parquet"
vitals_labeled.write_parquet(output_path)
log(f"Guardado en: {output_path}")

sepsis_stays = vitals_labeled.filter(pl.col("sepsis") == 1)["stay_id"].n_unique()
no_sep_stays = vitals_labeled.filter(pl.col("sepsis") == 0)["stay_id"].n_unique()
total_stays  = vitals_labeled["stay_id"].n_unique()

print("=" * 50)
print("  VITALS — COMPLETADO")
print("=" * 50)
print(f"  Stays totales:          {total_stays:,}")
print(f"  Stays con sepsis:       {sepsis_stays:,}")
print(f"  Stays sin sepsis:       {no_sep_stays:,}")
print(f"  Filas totales:          {len(vitals_labeled):,}")
print(f"  Media horas/stay:       {len(vitals_labeled)/total_stays:.1f}")
print(f"  Archivo: {output_path}")

[19:39:03] Uniendo con etiquetas de cohorte ...


[19:39:04] Guardado en: ../data/processed/vitals/vitals_hourly.parquet
  VITALS — COMPLETADO
  Stays totales:          74,829
  Stays con sepsis:       17,189
  Stays sin sepsis:       57,640
  Filas totales:          7,940,236
  Media horas/stay:       106.1
  Archivo: ../data/processed/vitals/vitals_hourly.parquet
